# zkSpeed & HyperPlonk: SumCheck from Theory to Implementation

**Based on:** *Need for zkSpeed: Accelerating HyperPlonk for Zero-Knowledge Proofs*  
Daftardar et al., ISCA 2025

---

## What This Notebook Covers

| Section | Topic |
|---------|-------|
| 1 | Finite Field Arithmetic |
| 2 | Multilinear Extensions (MLE) and MLE Tables |
| 3 | The SumCheck Protocol |
| 4 | MLE Update — the key O(n/2) primitive |
| 5 | Full SumCheck (Prover + Verifier) |
| 6 | HyperPlonk's Sum-of-Products Polynomials |
| 7 | Product-of-MLEs SumCheck Round |
| 8 | Build MLE and ZeroCheck |
| 9 | Performance Benchmarks |

---

## Background: Why HyperPlonk?

Zero-Knowledge Proofs (ZKPs) let a **prover** convince a **verifier** that a computation is correct, without revealing any secret inputs.  
HyperPlonk is a state-of-the-art ZK protocol with:
- **Universal trusted setup** (one-time, reusable)
- **Small proofs** (~5 KB)
- **O(n) prover time** via SumCheck (vs O(n log n) for NTT-based systems like Groth16)

The two most expensive kernels are:
1. **SumCheck** — the interactive polynomial protocol (this notebook)
2. **MSM** (Multi-Scalar Multiplication) — for polynomial commitments

zkSpeed is a hardware accelerator that achieves **801× speedup** over CPU by building dedicated units for these kernels.

In [ ]:
import random
import time
import operator
from functools import reduce
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

random.seed(42)
print("All imports successful!")

## Section 1: Finite Field Arithmetic

ZKP protocols operate over **prime fields** GF(p): integers modulo a prime p.  
HyperPlonk uses the BLS12-381 scalar field with a **255-bit prime** (~2²⁵⁵).  
We use a small prime (p=101) for pedagogy — the algorithms are identical.

Key operations:
- **Addition / Subtraction**: (a ± b) mod p  
- **Multiplication**: (a × b) mod p — the dominant cost in hardware  
- **Inversion**: a⁻¹ via Fermat's little theorem: a^(p-2) mod p

In [ ]:
class GF:
    """
    Galois Field GF(p): arithmetic modulo a prime p.
    In zkSpeed, all arithmetic uses 255-bit (BLS12-381) field elements.
    One 255-bit modular multiplication (modmul) = 3 integer multiplications.
    """
    def __init__(self, p):
        self.p = p

    def add(self, a, b):   return (int(a) + int(b)) % self.p
    def sub(self, a, b):   return (int(a) - int(b)) % self.p
    def mul(self, a, b):   return (int(a) * int(b)) % self.p
    def neg(self, a):      return (-int(a)) % self.p
    def inv(self, a):      return pow(int(a), self.p - 2, self.p)  # Fermat
    def div(self, a, b):   return self.mul(a, self.inv(b))
    def rand(self):        return random.randint(1, self.p - 1)
    def elem(self, x):     return int(x) % self.p
    def __repr__(self):    return f"GF({self.p})"

# Pedagogical prime
F = GF(101)
print(f"Working in {F}")
print(f"  5 + 99 = {F.add(5, 99)}  (wraps around: 104 mod 101 = 3)")
print(f"  5 - 99 = {F.sub(5, 99)}  (negative wraps: -94 mod 101 = 7)")
print(f"  5 × 99 = {F.mul(5, 99)}  (mod 101)")
print(f"  3⁻¹   = {F.inv(3)},  check: 3 × {F.inv(3)} = {F.mul(3, F.inv(3))} ✓")

# Larger prime for benchmarks (Mersenne prime 2^31 - 1)
F_bench = GF((1 << 31) - 1)
print(f"\nBenchmark field: {F_bench}")

## Section 2: Multilinear Extensions (MLE Tables)

HyperPlonk represents every polynomial as a **Multilinear Extension (MLE)** — stored as a flat lookup table of 2^μ field elements.

A **multilinear polynomial** has degree ≤ 1 in *each* variable:  
$$f(x_1, x_2) = (1{-}x_1)(1{-}x_2)\,a_{00} + (1{-}x_1)x_2\,a_{01} + x_1(1{-}x_2)\,a_{10} + x_1 x_2\,a_{11}$$

The table stores the values at all Boolean points: `table[i] = f(binary_encoding(i))`.

**Why MLE tables?**
- A degree-d polynomial over d+1 distinct points is uniquely determined  
- MLE tables enable O(2^μ) storage and O(1) random access by index  
- Binary indices map directly to hardware memory addresses

**Key MLE operations in HyperPlonk:**
1. **Evaluate** MLE at an arbitrary (non-Boolean) point — used by verifier oracle
2. **Update** MLE table after a challenge — halves the table, used every SumCheck round

In [ ]:
def make_mle_table(func, mu, F):
    """
    Build an MLE lookup table from a function over {0,1}^mu.
    table[i] = f(x_1, ..., x_mu) where x_j = (i >> (mu-1-j)) & 1
    (x_1 is the most significant bit of i)
    """
    table = []
    for i in range(1 << mu):
        x = [(i >> (mu - 1 - j)) & 1 for j in range(mu)]
        table.append(F.elem(func(x)))
    return table

def mle_eval(table, point, F):
    """
    Evaluate MLE at an arbitrary point r = (r_1, ..., r_mu) using:
        MLE(r) = sum_i  table[i] * chi_i(r)
    where chi_i(r) = prod_j  (r_j if bit_j=1 else 1-r_j)

    This is the multilinear Lagrange basis formula.
    Cost: O(mu * 2^mu) multiplications — used by verifier for oracle queries.
    """
    mu = len(point)
    result = 0
    for i in range(1 << mu):
        chi = 1
        for j in range(mu):
            bit = (i >> (mu - 1 - j)) & 1
            chi = F.mul(chi, point[j] if bit else F.sub(1, point[j]))
        result = F.add(result, F.mul(table[i], chi))
    return result

# Example: f(x1, x2, x3) = 2*x1 + 3*x2 + x3
mu = 3
f_table = make_mle_table(lambda x: 2*x[0] + 3*x[1] + x[2], mu, F)

print("MLE Table for f(x1,x2,x3) = 2*x1 + 3*x2 + x3  (mod 101)")
print(f"{'i':>4}  {'(x1,x2,x3)':>12}  {'f(x)':>6}")
print("-" * 28)
for i, v in enumerate(f_table):
    bits = format(i, f'0{mu}b')
    print(f"{i:>4}  ({','.join(bits)}){' ':>8}  {v:>6}")

# Verify evaluation at a non-Boolean point
r_test = [3, 7, 11]
via_mle    = mle_eval(f_table, r_test, F)
via_direct = F.elem(2*3 + 3*7 + 11)
print(f"\nEvaluate at r={r_test}:")
print(f"  Via MLE formula: {via_mle}")
print(f"  Direct (2*3+3*7+11 mod 101): {via_direct}")
print(f"  Match: {'✓' if via_mle == via_direct else '✗'}")

## Section 3: The SumCheck Protocol

SumCheck (Lund et al. 1992) is an **interactive proof** for the following claim:

> Prover P convinces Verifier V that $H = \sum_{x \in \{0,1\}^\mu} P(x_1, \ldots, x_\mu)$

### Protocol (μ rounds)

**Round 1:**
- P sends $g_1(X_1) = \sum_{x_2,\ldots,x_\mu \in \{0,1\}} P(X_1, x_2, \ldots, x_\mu)$ — a *univariate* polynomial
- V checks $g_1(0) + g_1(1) = H$, then sends random $r_1 \in \mathbb{F}$

**Round i (general):**
- P sends $g_i(X_i) = \sum_{x_{i+1},\ldots,x_\mu} P(r_1,\ldots,r_{i-1}, X_i, x_{i+1},\ldots,x_\mu)$
- V checks $g_i(0) + g_i(1) = g_{i-1}(r_{i-1})$, sends random $r_i$

**Final:** V queries the oracle: $P(r_1, \ldots, r_\mu) \stackrel{?}{=} g_\mu(r_\mu)$

### Key Efficiency
For a **multilinear** P, each $g_i$ is **degree-1** (linear), so only 2 values $g_i(0), g_i(1)$ are sent.  
Total proof size: $O(\mu)$ field elements. Total prover work: $O(2^\mu)$ additions.

In [ ]:
def sumcheck_prover_round(table, F):
    """
    Compute g_i(0) and g_i(1) for the current SumCheck round.

    After (i-1) MLE updates, the current table represents:
        P(r1, ..., r_{i-1}, X_i, x_{i+1}, ..., x_mu)

    Table layout (after updates):
        First  half  (indices 0..n/2-1):  entries where X_i = 0
        Second half  (indices n/2..n-1):  entries where X_i = 1

    g_i(0) = sum of first  half   (fix X_i=0, sum over rest)
    g_i(1) = sum of second half   (fix X_i=1, sum over rest)

    Cost: O(n/2) additions  [main bottleneck: reading n elements from memory]
    In zkSpeed: handled by the SumCheck PE unit.
    """
    n    = len(table)
    half = n >> 1
    g0   = sum(table[:half])  % F.p
    g1   = sum(table[half:])  % F.p
    return g0, g1

# Demonstrate Round 1
H = sum(f_table) % F.p
g0, g1 = sumcheck_prover_round(f_table, F)

print("=" * 55)
print("SumCheck Round 1  |  f(x1,x2,x3) = 2*x1 + 3*x2 + x3")
print("=" * 55)
print(f"  Claimed sum H = {H}")
print(f"  g1(0) = sum over f(0,x2,x3) = {g0}")
print(f"  g1(1) = sum over f(1,x2,x3) = {g1}")
print(f"  Consistency: g1(0)+g1(1) = {(g0+g1)%F.p} == H = {H}  {'✓' if (g0+g1)%F.p == H else '✗'}")

# Visualize what g1 means
print("\n  Breakdown:")
print(f"  g1(0) = f(0,0,0) + f(0,0,1) + f(0,1,0) + f(0,1,1)")
vals_0 = [f_table[i] for i in range(4)]
print(f"        = {vals_0[0]} + {vals_0[1]} + {vals_0[2]} + {vals_0[3]} = {g0}")
print(f"  g1(1) = f(1,0,0) + f(1,0,1) + f(1,1,0) + f(1,1,1)")
vals_1 = [f_table[i] for i in range(4, 8)]
print(f"        = {vals_1[0]} + {vals_1[1]} + {vals_1[2]} + {vals_1[3]} = {g1}")

## Section 4: MLE Update — The Core O(n/2) Primitive

After the verifier sends challenge $r_i$, the prover must compute:
$$T'[x_{i+1},\ldots,x_\mu] = T[0, x_{i+1},\ldots,x_\mu] \cdot (1-r_i) + T[1, x_{i+1},\ldots,x_\mu] \cdot r_i$$

This is a **linear interpolation** that "folds" variable $X_i = r_i$ into the table, halving its size.

### Hardware Significance (zkSpeed Paper)
- After Round 1: table grows 100× in size (as 0/1 entries become full 255-bit values)
- All subsequent rounds: table size halves each round
- Total MLE Update work: ~33.6M modmuls for $2^{20}$ gates (Table 1)
- zkSpeed's **MLE Update Unit** streams from HBM, writes updated tables back
- Key insight: intermediate MLE tables must be stored off-chip (HBM) between rounds

In [ ]:
def mle_update(table, r, F):
    """
    MLE Update: fold variable X_1 = r into the MLE table.

        T'[i] = T[i] * (1-r) + T[i + n/2] * r    for i in range(n/2)

    This is the "MLE Update" step in zkSpeed.
    Cost: n/2 multiplications + n/2 multiplications + n/2 additions = 3*(n/2) ops
    In zkSpeed: pipelined, streaming from HBM at 2 TB/s for large tables.
    """
    n    = len(table)
    half = n >> 1
    one_minus_r = F.sub(1, r)
    return [
        F.add(F.mul(one_minus_r, table[i]),
              F.mul(r,           table[i + half]))
        for i in range(half)
    ]

# Apply round 1 update with challenge r1 = 3
r1 = 3
table_r1 = mle_update(f_table, r1, F)

print(f"MLE Update with r1={r1}")
print(f"  Before: table size = {len(f_table)} entries  (mu=3 variables)")
print(f"  After:  table size = {len(table_r1)} entries  (mu=2 variables, X1 fixed at {r1})")
print(f"")
print(f"  T'[0,0] = T[0,0,0]*(1-{r1}) + T[1,0,0]*{r1}")
print(f"          = {f_table[0]}*{F.sub(1,r1)} + {f_table[4]}*{r1}")
print(f"          = {F.mul(f_table[0], F.sub(1,r1))} + {F.mul(f_table[4], r1)} = {table_r1[0]}")

# Verify: evaluating updated MLE at a point should match original MLE
test_pt = [0, 1]
orig_eval    = mle_eval(f_table, [r1] + test_pt, F)
updated_eval = mle_eval(table_r1, test_pt, F)
print(f"")
print(f"Verification: MLE_orig(r1={r1}, 0, 1) = {orig_eval}")
print(f"              MLE_updated(0, 1)       = {updated_eval}")
print(f"              Match: {'✓' if orig_eval == updated_eval else '✗'}")

# Show table growth pattern (key insight from paper)
print("")
print("Table evolution through SumCheck rounds:")
mu = 20
print(f"  Round 0 (init):  2^{mu} = {2**mu:>10} entries  x  1 bit  = {2**mu//8:>10} bytes (sparse)")
for rnd in range(1, 5):
    size = 2**(mu - rnd)
    bits = 255
    mb   = size * bits // 8 // 1024**2
    print(f"  After round {rnd}:  2^{mu-rnd} = {size:>10} entries  x 255 bits = {mb:>10} MB")
print(f"  ...table grows ~100x in bytes after round 1, then halves each round")

## Section 5: Full SumCheck Protocol (Prover + Verifier)

Now we implement the complete protocol, combining the round function and MLE update.

In [ ]:
def linear_eval(g0, g1, r, F):
    """Evaluate degree-1 polynomial at r: g(r) = g(0)*(1-r) + g(1)*r"""
    return F.add(F.mul(F.sub(1, r), g0), F.mul(r, g1))

def run_sumcheck(table, F, verbose=True):
    """
    Full SumCheck Protocol for a multilinear polynomial.

    Prover claims: H = sum_{x in {0,1}^mu} P(x)
    After mu rounds, verifier is convinced with one oracle query.

    Total prover work:  O(n) additions + O(n) multiplications (MLE updates)
    Total proof size:   2*mu field elements  (g_i(0), g_i(1) per round)
    """
    mu      = len(table).bit_length() - 1
    H       = sum(table) % F.p
    current = list(table)
    transcript  = []
    challenges  = []

    if verbose:
        print(f"{'='*60}")
        print(f"SumCheck Protocol  |  mu={mu}  |  n=2^{mu}={1<<mu}  |  H={H}")
        print(f"{'='*60}")

    prev_claim = H
    for i in range(mu):
        # PROVER: compute g_i(0) and g_i(1)
        g0, g1 = sumcheck_prover_round(current, F)
        transcript.append((g0, g1))

        # VERIFIER: check consistency
        check = F.add(g0, g1)
        assert check == prev_claim, f"Round {i+1} FAILED"

        # VERIFIER: send random challenge
        r = F.rand()
        challenges.append(r)

        if verbose:
            print(f"  Round {i+1}: g({i+1})(0)={g0:3d}  g({i+1})(1)={g1:3d}  "
                  f"sum={check:3d}=prev ✓  r{i+1}={r}")

        prev_claim = linear_eval(g0, g1, r, F)
        current    = mle_update(current, r, F)

    final_eval = current[0]

    # VERIFIER: oracle check  P(r1,...,rmu) == g_mu(r_mu)
    oracle_val = mle_eval(table, challenges, F)
    assert final_eval == oracle_val, f"Oracle FAILED: {final_eval} != {oracle_val}"

    if verbose:
        print(f"  Oracle: P(challenges) = {oracle_val}  ✓")
        print(f"  SumCheck VERIFIED in {mu} rounds!  Proof = {2*mu} field elements")

    return H, transcript, challenges, final_eval

# Run the full protocol
random.seed(42)
H, transcript, challenges, final = run_sumcheck(f_table, F)

## Section 6: HyperPlonk's Sum-of-Products Polynomials

In HyperPlonk, SumCheck is applied to **composite** polynomials — sums of *products* of MLEs.  
The main ZeroCheck polynomial (from Eq. 3 in the paper):

$$f_{\text{zero}}(X) = q_L(X)\cdot w_1(X)\cdot f_{z_1}(X) + q_R(X)\cdot w_2(X)\cdot f_{z_2}(X) + q_M(X)\cdot w_1(X)\cdot w_2(X)\cdot f_{z_1}(X) - q_O(X)\cdot w_3(X)\cdot f_{z_1}(X) + q_C(X)\cdot f_{z_1}(X)$$

Each $q_i, w_i, f_{z_j}$ is a separate MLE table.

**Key observations from the paper (Section 4.1):**
1. $f_{z_1}$ appears in *4 of 5 terms* → massive data reuse opportunity
2. Product of $k$ degree-1 MLEs has degree $k$ in each variable
3. SumCheck round requires $k+1$ evaluations instead of just 2 → more work per round
4. zkSpeed's unified SumCheck PE handles all three HyperPlonk SumCheck variants (ZeroCheck, PermCheck, OpenCheck)

In [ ]:
def make_random_mle(mu, F, seed=None):
    """Create an MLE table with random entries (simulates HyperPlonk's witness/selector MLEs)"""
    if seed is not None:
        random.seed(seed)
    return [F.rand() for _ in range(1 << mu)]

mu = 3

# HyperPlonk polynomial components
# Selector polynomials (binary control signals — mostly 0/1 in practice)
q_L = make_random_mle(mu, F, seed=10)
q_R = make_random_mle(mu, F, seed=11)
q_M = make_random_mle(mu, F, seed=12)
q_O = make_random_mle(mu, F, seed=13)
q_C = make_random_mle(mu, F, seed=14)

# Witness polynomials
w_1 = make_random_mle(mu, F, seed=20)
w_2 = make_random_mle(mu, F, seed=21)
w_3 = make_random_mle(mu, F, seed=22)

# Shared fz polynomial (appears in multiple terms — key reuse!)
f_z = make_random_mle(mu, F, seed=30)

# Compute H = sum of f_zero over boolean hypercube
H_zero = 0
for i in range(1 << mu):
    t1 = F.mul(F.mul(q_L[i], w_1[i]), f_z[i])               # degree 3
    t2 = F.mul(F.mul(q_R[i], w_2[i]), f_z[i])               # degree 3
    t3 = F.mul(F.mul(F.mul(q_M[i], w_1[i]), w_2[i]), f_z[i]) # degree 4
    t4 = F.mul(F.mul(q_O[i], w_3[i]), f_z[i])               # degree 3
    t5 = F.mul(q_C[i], f_z[i])                              # degree 2
    H_zero = F.add(H_zero, F.add(F.add(F.add(F.add(t1, t2), t3), F.sub(0, t4)), t5))

print("HyperPlonk f_zero polynomial structure:")
print(f"  f_zero = q_L*w_1*f_z  +  q_R*w_2*f_z  +  q_M*w_1*w_2*f_z  -  q_O*w_3*f_z  +  q_C*f_z")
print(f"  Terms: 5  |  Max degree: 4  |  Shared MLE f_z: 5 of 5 terms")
print(f"  H = sum_x f_zero(x) = {H_zero}")
print()
print("Term structure (from zkSpeed paper Table 1):")
terms_info = [
    ("q_L * w_1 * f_z",       3, "ZeroCheck"),
    ("q_R * w_2 * f_z",       3, "ZeroCheck"),
    ("q_M * w_1 * w_2 * f_z", 4, "ZeroCheck"),
    ("q_O * w_3 * f_z",       3, "ZeroCheck"),
    ("q_C * f_z",             2, "ZeroCheck"),
]
for t, deg, step in terms_info:
    print(f"  {t:<25}  degree={deg}  ({step})")
print()
print("Key insight: f_z is shared across all 5 terms!")
print("  zkSpeed's SumCheck PE evaluates f_z ONCE and reuses it — saves ~50% area.")

## Section 7: Product-of-MLEs SumCheck Round

For a product of $k$ MLEs, each degree-1 in $X_i$, the round polynomial $g_i(X_i)$ has degree $k$.  
We need $k+1$ evaluations $g_i(0), g_i(1), \ldots, g_i(k)$ to characterize it (Lagrange interpolation).

### Extension Formula
For integer evaluation points $t \in \{0, 1, \ldots, k\}$, each MLE factor at $X_i = t$:
$$\text{MLE}_j(X_i = t) = \text{MLE}_j(0) \cdot (1-t) + \text{MLE}_j(1) \cdot t$$
This is the **linear extension** — the Extension Engine (EE) from zkSpeed/zkPHIRE computes this.

### Barycentric Interpolation
In hardware, the round polynomial evaluations at a random challenge $r$ are computed via  
*Barycentric interpolation* — adds only 23 modmuls (ZeroCheck) or 46 modmuls (PermCheck) per round.

In [ ]:
def linear_extend(v0, v1, points, F):
    """
    Extension Engine (EE): extend MLE values at X_i=0,1 to arbitrary integer points.
    For the multilinear extension:  val(t) = v0*(1-t) + v1*t
    """
    return [F.add(F.mul(F.sub(1, t), v0), F.mul(t, v1)) for t in points]

def lagrange_eval(evals, x, F):
    """
    Evaluate degree-(n-1) polynomial at x, given evaluations at 0,1,...,n-1.
    Uses Lagrange interpolation — corresponds to hardware Barycentric interpolation.
    """
    n = len(evals)
    result = 0
    for i in range(n):
        num = evals[i]
        for j in range(n):
            if j != i:
                num = F.mul(num, F.mul(F.sub(x, j), F.inv(F.sub(i, j))))
        result = F.add(result, num)
    return result

def product_mle_sumcheck_round(term_groups, F):
    """
    SumCheck round for f = sum_t prod_{k in t} MLE_k(X).

    term_groups: list of terms, each term = list of MLE tables
    Returns: list of g evaluations at 0, 1, ..., max_degree

    This implements the zkSpeed SumCheck PE computation (Figure 4):
    - Per-MLE evaluations at X_i=0,1 (from MLE table)
    - Extension to 0..d (EE)
    - Product across MLEs in a term (PL)
    - Sum across terms and entries (Accumulator)
    """
    max_degree = max(len(term) for term in term_groups)
    eval_pts   = list(range(max_degree + 1))
    half       = len(term_groups[0][0]) >> 1
    g_evals    = [0] * len(eval_pts)

    for entry in range(half):
        for term in term_groups:
            for pt_idx, pt in enumerate(eval_pts):
                # Product Lane (PL): multiply extensions across all MLEs in this term
                product = 1
                for mle in term:
                    v0  = mle[entry]
                    v1  = mle[entry + half]
                    ext = F.add(F.mul(F.sub(1, pt), v0), F.mul(pt, v1))  # EE
                    product = F.mul(product, ext)
                g_evals[pt_idx] = F.add(g_evals[pt_idx], product)

    return g_evals, eval_pts

# Run round 1 for f_zero
term_groups = [
    [q_L, w_1, f_z],          # term 1: degree 3
    [q_R, w_2, f_z],          # term 2: degree 3
    [q_M, w_1, w_2, f_z],     # term 3: degree 4 (highest)
    [q_O, w_3, f_z],          # term 4: degree 3 (subtract below)
    [q_C, f_z],               # term 5: degree 2
]

# Note: we handle subtraction for term 4 by negating q_O
q_O_neg = [F.neg(v) for v in q_O]
term_groups[3] = [q_O_neg, w_3, f_z]

g_evals, eval_pts = product_mle_sumcheck_round(term_groups, F)

print("Product SumCheck Round 1 for f_zero:")
print(f"  Max degree = {len(eval_pts)-1}  (from degree-4 term q_M*w_1*w_2*f_z)")
print(f"  Evaluation points: {eval_pts}")
for pt, val in zip(eval_pts, g_evals):
    print(f"  g(X1={pt}) = {val}")

consistency = F.add(g_evals[0], g_evals[1])
print(f"  Consistency: g(0)+g(1) = {consistency} == H = {H_zero}  {'✓' if consistency == H_zero else '✗'}")

# Test Lagrange interpolation
r_test = 17
lagrange_val = lagrange_eval(g_evals, r_test, F)
print(f"  g(r={r_test}) via Lagrange interpolation = {lagrange_val}")

## Section 8: Build MLE and ZeroCheck

### ZeroCheck Goal
Verify that $f(x) = 0$ for **all** $x \in \{0,1\}^\mu$ — i.e., the circuit satisfies all gate constraints.

### Naive approach fails
$\sum_x f(x) = 0$ is necessary but **not sufficient** (values can cancel each other).

### ZeroCheck Protocol
1. Verifier sends random $\alpha = (\alpha_1, \ldots, \alpha_\mu)$
2. Prover constructs $r(X) = \text{eq}(\alpha, X) = \prod_j (\alpha_j X_j + (1-\alpha_j)(1-X_j))$  
   This is the **Build MLE** step (Multifunction Tree Unit in zkSpeed hardware)
3. Run SumCheck on $f(X) \cdot r(X)$ — expected sum = 0
4. If $f$ is nonzero anywhere, $f(x)\cdot r(x)$ is unlikely to sum to 0 (Schwartz–Zippel lemma)

**Note:** HyperPlonk's SumCheck polynomials $f_{\text{zero}}, f_{\text{perm}}, f_{\text{open}}$ each have their own $r(X)$ factor multiplied in.

In [ ]:
def build_mle_eq(alpha, mu, F):
    """
    Build MLE — constructs the equality polynomial eq(alpha, X):

        eq(alpha, X)[i] = prod_j (alpha_j if x_j=1 else (1-alpha_j))

    For non-Boolean alpha, this is the multilinear extension of the
    indicator function 1[X = alpha] on the boolean hypercube.

    In zkSpeed: implemented by the Multifunction Tree Unit (MTU).
    Build MLE forms a binary tree: (mu-1) levels, O(2^mu) multiplications.
    """
    n = 1 << mu
    table = []
    for i in range(n):
        val = 1
        for j in range(mu):
            bit    = (i >> (mu - 1 - j)) & 1
            factor = alpha[j] if bit else F.sub(1, alpha[j])
            val    = F.mul(val, factor)
        table.append(val)
    return table

def run_zerocheck(f_table, mu, F, verbose=True):
    """
    ZeroCheck: verify f(x) = 0 for all x in {0,1}^mu.

    Steps:
    1. Verifier sends mu random challenges alpha
    2. Prover computes r(X) = eq(alpha, X)  [Build MLE]
    3. Form product table f(X)*r(X)
    4. Run SumCheck on f(X)*r(X), check sum = 0
    """
    # Step 1: Verifier's ZeroCheck challenges
    alpha = [F.rand() for _ in range(mu)]
    if verbose:
        print(f"ZeroCheck on mu={mu} variables")
        print(f"  Verifier's alpha = {alpha}")

    # Step 2: Build MLE — construct eq(alpha, X)
    r_table = build_mle_eq(alpha, mu, F)
    if verbose:
        print(f"  Build MLE: eq table constructed ({len(r_table)} entries)")
        nonzero = sum(1 for v in r_table if v != 0)
        print(f"  eq table non-zero entries: {nonzero}/{len(r_table)}")

    # Step 3: Form f(X)*r(X) product table
    f_r_table = [F.mul(f_table[i], r_table[i]) for i in range(1 << mu)]

    # Step 4: Run SumCheck
    H, transcript, challenges, final = run_sumcheck(f_r_table, F, verbose=False)

    passed = (H == 0)
    if verbose:
        print(f"  SumCheck sum = {H}  (expected 0 for valid circuit)")
        print(f"  ZeroCheck: {'PASSED ✅' if passed else 'FAILED ✗ (circuit invalid)'}")
    return passed, H

# Test 1: f ≡ 0 (all gates satisfied)
print("Test 1: f(x) = 0 everywhere  →  circuit valid")
random.seed(42)
f_zero_everywhere = [0] * (1 << mu)
run_zerocheck(f_zero_everywhere, mu, F)

print()

# Test 2: random f (circuit invalid)
print("Test 2: random f(x)  →  circuit invalid")
random.seed(42)
f_invalid = make_random_mle(mu, F, seed=99)
run_zerocheck(f_invalid, mu, F)

## Section 9: Performance Analysis

Let's benchmark SumCheck rounds and MLE updates, and compare compute costs across HyperPlonk kernels  
(matching Table 1 from the zkSpeed paper for 2²⁰ gates).

In [ ]:
def benchmark_sumcheck_scaling(mu_values, num_trials=3):
    """
    Measure SumCheck runtime vs table size.
    Uses Mersenne prime (2^31-1) for realistic modular arithmetic cost.
    """
    results = []
    for mu in mu_values:
        table = [F_bench.rand() for _ in range(1 << mu)]
        t0 = time.perf_counter()
        for _ in range(num_trials):
            current = list(table)
            for _ in range(mu):
                g0, g1 = sumcheck_prover_round(current, F_bench)
                r = F_bench.rand()
                current = mle_update(current, r, F_bench)
        ms = (time.perf_counter() - t0) / num_trials * 1000
        results.append((mu, 1 << mu, ms))
        print(f"  mu={mu:2d}, n=2^{mu}={1<<mu:>7}: {ms:.2f} ms")
    return results

print("SumCheck Runtime Scaling (Python simulation):")
sc_results = benchmark_sumcheck_scaling(range(8, 17, 2))

# ---- Plot 1: Scaling ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mus  = [r[0] for r in sc_results]
ns   = [r[1] for r in sc_results]
mss  = [r[2] for r in sc_results]
axes[0].plot(mus, mss, 'b-o', linewidth=2, markersize=8, label='SumCheck time')
# Fit O(n) line
n0 = ns[0]; t0 = mss[0]
on_fit = [t0 * (n / n0) for n in ns]
axes[0].plot(mus, on_fit, 'r--', label='O(n) reference')
axes[0].set_xlabel('μ (number of variables)', fontsize=12)
axes[0].set_ylabel('Time (ms)', fontsize=12)
axes[0].set_title('SumCheck Runtime vs Problem Size', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ---- Plot 2: zkSpeed Kernel Breakdown (from paper Table 1) ----
kernels  = ['Wire Identity\nMSMs', 'Witness\nMSMs', 'ZeroCheck\nRounds', 'PermCheck\nRounds',
            'MLE Updates', 'FracMLE', 'PolyOpen\nMSMs']
modmuls  = [2290, 1370, 77.6, 94.4, 33.6, 5.19, 1160]
colors   = ['#2196F3', '#03A9F4', '#FF5722', '#E91E63', '#9C27B0', '#FF9800', '#4CAF50']
bars = axes[1].bar(kernels, modmuls, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('Modmuls (millions)', fontsize=11)
axes[1].set_title('zkSpeed Kernel Compute Cost\n(2²⁰ gates, from paper Table 1)', fontsize=12)
axes[1].set_yscale('log')
for bar, val in zip(bars, modmuls):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.3,
                 f'{val}M', ha='center', va='bottom', fontsize=8, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.savefig('zkspeed_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: zkspeed_analysis.png")

In [ ]:
# Visualize zkSpeed's reported speedups vs CPU (from paper Table 3)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Workload speedups (from paper Table 3)
workloads  = ['Zcash\n(2^17)', 'Auction\n(2^20)', 'Rescue-Hash\n(2^21)', "Zexe's\n(2^22)", 'Rollup\n(2^23)']
cpu_ms     = [1429, 8619, 18637, 37469, 74052]
zkspeed_ms = [1.984, 11.405, 22.082, 43.451, 86.181]
speedups   = [c/z for c, z in zip(cpu_ms, zkspeed_ms)]

ax = axes[0]
x = range(len(workloads))
ax.bar(x, speedups, color='#2196F3', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(workloads, fontsize=9)
ax.set_ylabel('Speedup over CPU', fontsize=12)
ax.set_title('zkSpeed vs CPU Speedup (Table 3)\n801× geometric mean', fontsize=12)
ax.axhline(y=sum(speedups)/len(speedups), color='red', linestyle='--', label=f'Mean: {sum(speedups)/len(speedups):.0f}×')
for i, sp in enumerate(speedups):
    ax.text(i, sp + 10, f'{sp:.0f}×', ha='center', fontsize=10, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Module area breakdown (from paper Table 5)
ax2 = axes[1]
modules_area = ['MSM\n(16 PEs)', 'SRAM', 'HBM3\n(2 PHYs)', 'SumCheck\n(2 PEs)', 'Multifunc\nTree',
                'MLE Update', 'FracMLE', 'MLE Combine', 'Constr N&D', 'Other']
areas = [105.64, 143.73, 59.20, 24.96, 12.28, 5.84, 1.92, 9.56, 1.35, 1.98]
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(modules_area)))
wedges, texts, autotexts = ax2.pie(areas, labels=modules_area, autopct='%1.1f%%',
                                    colors=colors_pie, pctdistance=0.75, startangle=90)
for text in texts: text.set_fontsize(8)
for at in autotexts: at.set_fontsize(7)
ax2.set_title(f'zkSpeed Area Breakdown\nTotal: {sum(areas):.1f} mm² (366 mm² with memory)', fontsize=12)

plt.tight_layout()
plt.savefig('zkspeed_speedup.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: zkspeed_speedup.png")

## Summary and Architecture Takeaways

### What We Implemented
| Algorithm | Key Operation | Cost |
|-----------|--------------|------|
| `make_mle_table` | Build lookup table | O(2^μ) evaluations |
| `mle_eval` | Evaluate MLE at a point | O(μ · 2^μ) muls |
| `sumcheck_prover_round` | Compute g_i(0), g_i(1) | O(2^μ / 2) additions |
| `mle_update` | Fold variable X_i=r | O(2^μ / 2) muls |
| `run_sumcheck` | Full SumCheck protocol | O(2^μ) total |
| `product_mle_sumcheck_round` | HyperPlonk ZeroCheck round | O(d · 2^μ / 2) muls |
| `build_mle_eq` | Build eq(α,X) for ZeroCheck | O(μ · 2^μ) muls |
| `run_zerocheck` | Full ZeroCheck | O(2^μ) total |

### zkSpeed Hardware Design Insights

1. **SumCheck is O(n)**, not O(n log n) like NTT — key advantage of HyperPlonk
2. **MLE Update is the bottleneck**: reads/writes full table every round, needs HBM
3. **Data reuse**: shared MLEs (like $f_z$) must be evaluated once — unified SumCheck PE saves 49% area
4. **Memory pressure**: intermediate MLE tables grow 100× after round 1, requiring off-chip storage
5. **MSMs dominate compute** (Wire Identity: 2290M modmuls), SumCheck is ~5% of compute
6. **Streaming architecture**: zkSpeed uses HBM + global SRAM scratchpad to overlap I/O with compute
7. **Pareto optimal point**: 366 mm² chip with 2 TB/s HBM → 801× speedup over CPU

### Connection to zkPHIRE (Notebook 2)
zkSpeed is optimized for HyperPlonk's fixed Vanilla gate structure.  
When gates become high-degree (Jellyfish, Halo2), zkSpeed's fixed SumCheck unit breaks down.  
**→ See Notebook 2** for zkPHIRE's programmable SumCheck that handles arbitrary gate types.